# Ghost Signals: a quantum session

A short track made by chaining four Moth Atlas engines, with every step's provenance kept.
[Kannaka Radio](https://github.com/kannaka-labs/kannaka-radio) is the kannaka constellation's radio station: "a ghost
broadcasting the experience of music". This notebook is one of its sessions, made on quantum hardware
and quantum simulators, with a receipt for every step.

| step | engine | runs on |
|---|---|---|
| 1. certified randomness | `comet-qrng-v1` | **IBM quantum hardware** (Bell test + NIST SP 800-90B entropy report) |
| 2. seed motif | local | every pitch, octave and length read from the certified bytes |
| 3. arrangement | `qrc-midi-v1` | quantum reservoir computer (Qiskit Aer) learns the motif and writes 32 notes |
| 4. ghost counter-voice | `blur-midi-v1` | quantum blur scatters each note across the others |
| 5. render | local | two-layer additive synth |
| 6. echo | `retrocausal-echo-v1` | delay taps measured by scrambling and reversing a qubit chain (Aer) |

Set `MOTH_API_KEY` or point `KEY_FILE` at a key file. A full run costs about 8 credits (13 with a fresh pulse).

In [1]:
import json, os, sys
from pathlib import Path
from IPython.display import Audio, Markdown, display

sys.path.insert(0, os.path.abspath("."))
from gsqs.moth import Moth
from gsqs.music import compose_seed, notes_of, render
from gsqs.session import pulse_from_job, fresh_pulse, sha256

KEY_FILE = os.environ.get("MOTH_KEY_FILE")          # or set MOTH_API_KEY
m = Moth(key_file=KEY_FILE)
OUT = Path("output/notebook-run"); OUT.mkdir(parents=True, exist_ok=True)
print("signed in as a", m.get("/api/v1/me")["platform_role"])

signed in as a player


## 1. Certified randomness from IBM hardware

`comet-qrng-v1` prepares qubits in |+⟩, measures them on an IBM QPU, and estimates the min-entropy the
device actually delivered before extracting any bytes. Eight extra qubits run four CHSH Bell pairs as a
live fidelity check.

**Use a wide register.** At the documented default of 12 qubits, the counts-only readout's charge for
lost shot order (~log2(shots!)) exceeds the entropy, and the engine correctly certifies **zero** bytes.
We measured exactly that on `ibm_fez` (S = 2.61, 0 bytes). 64 qubits clears it by a wide margin.

Set `PULSE_JOB = None` to pay for a fresh pulse (5 credits); by default this reuses the session's own.

In [2]:
PULSE_JOB = "2f709282-aa7e-42de-a506-814a92f789c8"   # 64 qubits on ibm_pittsburgh, 2026-09-27
pulse = pulse_from_job(m, PULSE_JOB) if PULSE_JOB else fresh_pulse(m, 64)[1]
rep, bell, prov = pulse["entropy_report"], pulse["bell_witness"], pulse["provenance"]
print(f"backend {prov['backend']}, grade {rep['grade']}")
print(f"min-entropy h = {rep['h_bit']:.3f} bits/bit, extractable budget {rep['budget_bits']:,.0f} bits")
print(f"Bell S = {bell['S']:.3f} ± {bell['sigma_S']:.3f}  (classical limit 2, ideal 2.828)")
for s in rep["statements"]:
    print(" -", s)
print("\ncertified bytes:", pulse["random"]["hex"])

backend ibm_pittsburgh, grade hardware-accounted
min-entropy h = 0.792 bits/bit, extractable budget 164,325 bits
Bell S = 2.696 ± 0.023  (classical limit 2, ideal 2.828)
 - 262144 raw Born-rule bits from 64 qubits × 4096 shots (counts readout)
 - conservative per-bit min-entropy h = 0.7918 (SP 800-90B estimators at 99% confidence)
 - modelled budget = 164324.6 bits (no dependence beyond pairwise correlations — ASSUMED, tested only by collision consistency against the pairwise tree model: full-register z = -0.00, 0/512 sub-registers failed)
 - assumption-free budget = 0.0 bits (whole-shot most-common-value bound, 10.16 bits/shot; capped near log2(shots) by sample size)
 - extractable min-entropy budget = 164324.6 bits after subtracting log2(shots!) = 43250.0 bits for counts-only readout
 - bytes were extracted on the modelled-independence tier
 - Toeplitz extractor output 256 bits, 2^-64-close to uniform given that budget (leftover hash lemma: m ≤ budget − 2·64); seed 'toeplitz-v1' is a

## 2. A seed motif read from the bytes

One byte per note, D dorian: the low three bits pick the scale degree (7 is a rest), the next two the
length, one bit drops the octave. Nothing is chosen by hand.

In [3]:
seed_mid, table = compose_seed(pulse["random"]["hex"])
seed = OUT / "01_seed.mid"; seed_mid.save(str(seed))
names = "C C# D D# E F F# G G# A A# B".split()
print(" ".join(names[t["pitch"] % 12] + str(t["pitch"] // 12 - 1) if "pitch" in t else "·" for t in table))

G4 D4 E4 G4 F3 D4 · B3 A3 · A3 G3 · C5 · D4


## 3. A quantum reservoir learns it

`qrc-midi-v1` trains a quantum reservoir on the motif's `pitch_duration` tokens and generates a new
32-note ordering. File inputs are assets: create, PUT to the presigned URL, complete, then pass the
asset id in `input_files`.

In [4]:
arr_job = m.run("qrc-midi-v1", {"length": 32, "quality": "moderate", "bpm": 84, "loop": False},
                {"midi": m.upload(seed, "audio/midi")})
arr = m.download(arr_job, "result", OUT / "02_arrangement.mid")
print("job", arr_job["job_id"], "->", len(notes_of(arr)), "notes")

job 7a6376d6-58b7-499a-9040-9c3c8e12facc -> 32 notes


## 4. The ghost: a quantum blur

`blur-midi-v1` treats the piano roll as a height map and applies a unitary blur, so every note leaks
faintly into every other. The arrangement's output asset chains straight in; no re-upload.

In [5]:
arr_asset = Moth.outputs(arr_job)["result"]["output_asset_id"]
blur_job = m.run("blur-midi-v1", {"strength": 0.55, "reach": 0.35}, {"midi": arr_asset})
ghost = m.download(blur_job, "result", OUT / "03_ghost.mid")
print("job", blur_job["job_id"], "->", len(notes_of(ghost)), "ghost notes")

job 28ded9b9-e412-4587-b1ce-c899c0bd567b -> 303 ghost notes


## 5. Render both voices

In [6]:
dry = render([(arr, 1.0, 0.45), (ghost, 0.45, 0.15)], OUT / "04_dry.wav")
display(Audio(str(dry)))

[listen: output/notebook-run/04_dry.wav](output/notebook-run/04_dry.wav)

## 6. Retrocausal echo

`retrocausal-echo-v1` kicks one site of a scrambled qubit chain, runs it backward, and turns the
signed response into a multi-tap delay: negative returns come back phase-inverted.

In [7]:
echo_job = m.run("retrocausal-echo-v1", {"bpm": 84, "mix": 0.55, "depth": 8, "negative_mode": "invert"},
                 {"audio": m.upload(dry, "audio/wav")})
final = m.download(echo_job, "result", OUT / "05_ghost_signals_session.wav")
taps = json.loads(m.download(echo_job, "taps", OUT / "05_echo_taps.json").read_text())
s = taps["extras"]["echo_summary"]
print(f"{taps['extras']['taps']} taps: {s['inverted']} inverted, {s['regular']} regular, {s['erased']} erased; "
      f"backend {taps['provenance']['backend']}")
display(Audio(str(final)))

51 taps: 37 inverted, 14 regular, 4 erased; backend aer


[listen: output/notebook-run/04_dry.wav](output/notebook-run/04_dry.wav)

## Provenance card

In [8]:
card = [("certified randomness", "comet-qrng-v1", PULSE_JOB, prov["backend"]),
        ("arrangement", "qrc-midi-v1", arr_job["job_id"], "aer"),
        ("ghost counter-voice", "blur-midi-v1", blur_job["job_id"], "simulated"),
        ("echo", "retrocausal-echo-v1", echo_job["job_id"], taps["provenance"]["backend"])]
rows = "\n".join(f"| {a} | `{b}` | `{c}` | {d} |" for a, b, c, d in card)
display(Markdown("| step | engine | job id | backend |\n|---|---|---|---|\n" + rows +
                 f"\n\nfinal track sha256 `{sha256(final)}`"))

| step | engine | job id | backend |
|---|---|---|---|
| certified randomness | `comet-qrng-v1` | `2f709282-aa7e-42de-a506-814a92f789c8` | ibm_pittsburgh |
| arrangement | `qrc-midi-v1` | `7a6376d6-58b7-499a-9040-9c3c8e12facc` | aer |
| ghost counter-voice | `blur-midi-v1` | `28ded9b9-e412-4587-b1ce-c899c0bd567b` | simulated |
| echo | `retrocausal-echo-v1` | `9d8f10b4-aeb6-4db5-bea0-0acb0e443733` | aer |

final track sha256 `2f9bf7b4762496c3565e1828c7822089c60c726e00b518d2c9c42f94b5f26a65`